### Now we are building the FULL SaaS VERSION in Colab (with subscription + usage limits) and still NO paid API.
We will use a Fake AI model so the system behaves like real SaaS.

Think of this as your final mini-startup project

## FINAL GOAL

You will run a real SaaS AI app in Google Colab with:

- Login/Register
- Multi-user database
- Chat history
- Free vs Pro plans
- Daily usage limits
- Upgrade button
- Public web links

## STEP 0 — Install Packages

Run in Colab:

In [1]:
!pip install streamlit flask pyngrok requests --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 60.2 MB/s eta 0:00:00


## STEP 1 — Create DATABASE (users + plans + usage + chats)

Run this cell:

In [2]:
%%writefile db.py
import sqlite3

def create_tables():
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    # USERS TABLE (with plan + usage)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS users(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT,
        password TEXT,
        plan TEXT DEFAULT 'free',
        usage_count INTEGER DEFAULT 0
    )
    """)

    # CHAT HISTORY TABLE
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS chats(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        user TEXT,
        message TEXT,
        reply TEXT
    )
    """)

    conn.commit()
    conn.close()

Writing db.py


## STEP 2 — Add USER + PLAN + USAGE FUNCTIONS

In [3]:
%%writefile auth.py
import sqlite3

# Register
def register_user(username, password):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO users(username,password) VALUES(?,?)",
        (username,password)
    )
    conn.commit()
    conn.close()

# Login
def login_user(username,password):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    cursor.execute(
        "SELECT * FROM users WHERE username=? AND password=?",
        (username,password)
    )
    result = cursor.fetchone()
    conn.close()
    return result

# Check usage limit
def can_use(user):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute(
        "SELECT plan, usage_count FROM users WHERE username=?",
        (user,)
    )
    plan, usage = cursor.fetchone()

    if plan == "free" and usage >= 5:
        return False
    return True

# Increase usage
def update_usage(user):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute(
        "UPDATE users SET usage_count = usage_count + 1 WHERE username=?",
        (user,)
    )
    conn.commit()
    conn.close()

# Upgrade to PRO (fake payment)
def upgrade_user(user):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()

    cursor.execute(
        "UPDATE users SET plan='pro' WHERE username=?",
        (user,)
    )
    conn.commit()
    conn.close()

Writing auth.py


### STEP 3 — Create AI BACKEND (Flask API)

This simulates GPT.

In [7]:
%%writefile api.py
from flask import Flask, request, jsonify
from db import create_tables
from auth import can_use, update_usage
import sqlite3

app = Flask(__name__)
create_tables()

def save_chat(user,message,reply):
    conn = sqlite3.connect("chat.db")
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO chats(user,message,reply) VALUES(?,?,?)",
        (user,message,reply)
    )
    conn.commit()
    conn.close()

@app.route("/chat", methods=["POST"])
def chat():
    data = request.json
    user = data.get("user_id")
    message = data.get("message")

    #  Usage limit check
    if not can_use(user):
        return jsonify({"response":" Daily limit reached. Upgrade to PRO."})

    #  Fake AI (replace later with OpenAI)
    reply = " AI: " + message[::-1]

    update_usage(user)
    save_chat(user,message,reply)

    return jsonify({"response":reply})

app.run(port=5000)

Overwriting api.py


Fake AI trick used:

👉 message[::-1] = reverse text (looks like AI response)


### STEP 4 — Connect NGROK (IMPORTANT)

Paste your ngrok token:
```
!ngrok config add-authtoken YOUR_TOKEN
```
Start API tunnel:
```
from pyngrok import ngrok
api_url = ngrok.connect(5000)
api_url
```
Copy the link.

If you want to use Cloudflare Tunnel in Colab:

Download the binary:

In [3]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-04-28 09:27:04--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.27.177.113
Connecting to github.com (github.com)|20.27.177.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-04-28 09:27:05--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-28T10%3A12%3A56Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4d

This installs the Cloudflared CLI tool.

Run a tunnel:

```
!cloudflared tunnel --url http://localhost:5000

```
This will print a public URL like:
```
https://randomstring.trycloudflare.com
```
Use that URL in your Streamlit frontend:
```
API_URL = "https://randomstring.trycloudflare.com"
```

In [7]:
!cloudflared tunnel --url http://localhost:5000

2026-04-28T09:36:34Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-28T09:36:34Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-28T09:36:37Z INF +--------------------------------------------------------------------------------------------+
2026-04-28T09:36:37Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-28T09:36:37Z INF |  https://classics-shareware-saves-premium.trycloudflar

#### To access your Flask app from outside Colab, you need to use the public tunnel URL that Colab gave you (via Localtunnel or Cloudflared). For example:

```
https://giant-jobs-rest.loca.lt
```
or
```
https://discussions-wise-donor-bali.trycloudflare.com
```
That’s the link you should open in your browser — not localhost:5000.


### Alternatives
If installing Cloudflared feels heavy, stick with Localtunnel:
```
!npm install -g localtunnel
!lt --port 5000
```

### Summary

- The error occurs because cloudflared isn’t a Python package, so pip can’t install it.

- Fix by downloading the binary (.deb) and installing it with dpkg.

- Or use Localtunnel for a lighter, token‑free solution.

### STEP 5 — Create STREAMLIT FRONTEND

Replace PASTE_API_URL with ngrok link.

In [6]:
%%writefile app.py
import streamlit as st
import requests
from auth import register_user, login_user, upgrade_user
from db import create_tables

create_tables()

API_URL = "PASTE_API_URL/chat"

st.title("ChatNest AI SaaS")

menu = ["Login","Register"]
choice = st.selectbox("Menu", menu)

# Register
if choice == "Register":
    user = st.text_input("Username")
    password = st.text_input("Password", type="password")

    if st.button("Register"):
        register_user(user,password)
        st.success("Registered!")

# Login
elif choice == "Login":
    user = st.text_input("Username")
    password = st.text_input("Password", type="password")

    if st.button("Login"):
        result = login_user(user,password)
        if result:
            st.session_state["user"] = user
            st.success("Logged in!")
        else:
            st.error("Invalid login")

# Chat Area
if "user" in st.session_state:
    st.write("Welcome", st.session_state["user"])

    if st.button("Upgrade to PRO 💳"):
        upgrade_user(st.session_state["user"])
        st.success("You are now PRO!")

    msg = st.text_input("Ask AI")

    if st.button("Send"):
        res = requests.post(API_URL,
            json={"user_id":st.session_state["user"],"message":msg}
        )
        st.write(res.json()["response"])

Writing app.py


### STEP 6 — Run BOTH SERVERS

Run Flask:
```
!python api.py &
```
Expose Streamlit:
```
streamlit_url = ngrok.connect(8501)
streamlit_url
```
Run Streamlit:
```
!streamlit run app.py &>/dev/null &

I BUILT A REAL MINI SAAS

Open Streamlit link and test:

1. Register user
2. Login
3. Ask AI → works 5 times
4. Hit limit
5. Click Upgrade to PRO
6. Unlimited usage

### WHAT I JUST BUILT (Industry Level)

Now understand:

- Authentication systems
- Usage limiting systems
- Freemium SaaS model
- Frontend + Backend architecture
- Public deployment

This is literally how AI startups start.